In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
# Q1: Number of lesson pages
len(files)

72

In [4]:
import json
print(json.dumps(documents[0],indent=2))

{
  "content": "# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we'll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type \"how are\" in WhatsApp, it suggests\n\"you\" as the next word. \"How are you\" is the most common continuation.\nYour phone use

In [5]:
# Q2 index documents with minsearch
from minsearch import Index

index=Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [6]:
query='How does the agentic loop keep calling the model until it stops?'

boost_dict={'content':3.0,'filename':1.0}
filter_dict={}

In [7]:
results=index.search(
    query=query,
    num_results=5,
    boost_dict=boost_dict,
    filter_dict=filter_dict
)

In [8]:
for result in results:
    print(result["filename"])

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/16-other-frameworks.md


In [9]:
# Q3 with revised RagBase class (based on filename and content),  how many input tokens did we send to the model
# open_api_key need to retrive from .env file
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

from rag_helper_hw import RAGBase

# initiate RAGBase with knowledge base index and llm_client
# the remaining args are pre-defined in modular file or have default values.
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [10]:
myanswer=assistant.rag(query)

In [11]:
print(myanswer[0].input_tokens,myanswer[0].output_tokens)

7136 114


In [12]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

cost = calculate_gpt54mini_price(myanswer[0].input_tokens,myanswer[0].output_tokens)
print("(Input_tokens, Output_tokens): ", myanswer[0].input_tokens,myanswer[0].output_tokens)
print("Total cost: $", round(cost["total_cost"], 8))

(Input_tokens, Output_tokens):  7136 114
Total cost: $ 0.0011388


In [15]:
print(myanswer[1])

It keeps calling the model inside a `while True` loop.

Each iteration:
- sends the full message history to the model,
- checks the response for any `function_call`,
- runs the tool if needed,
- appends the tool result to the messages,
- and repeats.

It stops when the model returns a response with **no function calls**. The exit condition is basically:

```python
if has_function_calls == False:
    break
```

So the model keeps being called until it gives a final answer without asking for more tools.


In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [14]:
len(chunks)

295

In [15]:
# Q5 Index the chunk instead of documents
index_chunk=index.fit(chunks)

In [16]:
assistant_chunk = RAGBase(
    index=index_chunk,
    llm_client=openai_client,
)

In [17]:
myanswer_chunk=assistant_chunk.rag(query)

In [18]:
cost_chunk = calculate_gpt54mini_price(myanswer_chunk[0].input_tokens,myanswer_chunk[0].output_tokens)
print("(Input_tokens, Output_tokens): ", myanswer_chunk[0].input_tokens,myanswer_chunk[0].output_tokens)
print("Total cost: $", round(cost["total_cost"], 8))

(Input_tokens, Output_tokens):  2319 95
Total cost: $ 0.0011388


In [19]:
def search_chunk(query: str) -> dict[str, str]:
    """
    Search the course database for entries matching the given query.
    """
    return index_chunk.search(
        query,
        num_results=5,
        boost_dict={"content": 3.0, "filename": 0.5},
        filter_dict={}
    )

In [20]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [21]:
agent_tools = Tools()
agent_tools.add_tool(search_chunk)

In [23]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search_chunk',
  'description': 'Search the course database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [24]:
instructions_chunk="""
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions_chunk,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [25]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
